In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [2]:
import random

import torch
import numpy as np
from torch.utils.data import Dataset
from torchvision.io import read_image

def scale_and_pad_to_square(image, size=512):
    _, h, w = image.shape
    scale = 512 / max(h, w)
    image = T.functional.resize(image, (int(h*scale), int(w*scale)))
    _, h, w = image.shape
    l = (512 - w) // 2
    r = 512 - w - l
    t = (512 - h) // 2
    b = 512 - h - t
    return T.functional.pad(image, (l, t, r, b))

def transform(sample, train_mode=False):
    if train_mode:
        # random horizontal flip
        if random.random() < 0.5:
            sample["full_masks"] = (
                T.functional.hflip(sample["full_masks"][0]),
                T.functional.hflip(sample["full_masks"][1])
            )
            sample["burn_masks"] = (
                T.functional.hflip(sample["burn_masks"][0]),
                T.functional.hflip(sample["burn_masks"][1])
            )

    # scale and pad to square
    if "full_masks" in sample:
        sample["full_masks"] = (
            scale_and_pad_to_square(sample["full_masks"][0]),
            scale_and_pad_to_square(sample["full_masks"][1])
        )
    if "burn_masks" in sample:
        sample["burn_masks"] = (
            scale_and_pad_to_square(sample["burn_masks"][0]),
            scale_and_pad_to_square(sample["burn_masks"][1])
        )
    return sample

class SAM_Precessed_MHB(Dataset):
    _views_per_burn = 4
    def __init__(self, root_dir, train_mode = False, transform=transform, return_full_masks=True, return_burn_masks=True):
        self.root_dir = root_dir
        self.train_mode = train_mode
        self.transform = transform
        if train_mode:
            self._pairs_per_burn = 1
        else:
            self._pairs_per_burn = self._views_per_burn ** 2
        self.labels = torch.tensor(np.load(os.path.join(root_dir, "labels.npy")), dtype=torch.float32)
        self.return_full_masks = return_full_masks
        self.return_burn_masks = return_burn_masks
        self._len = self.labels.numel() * self._pairs_per_burn
        self._burn_levels = self.labels.shape[1]
        self._pairs_per_human = self._burn_levels * self._pairs_per_burn

    def __len__(self):
        return self._len

    def __getitem__(self, idx):
        human_idx = idx // self._pairs_per_human
        burn_idx = idx % self._pairs_per_human // self._pairs_per_burn
        if self.train_mode:
            front_view_idx = random.randint(0, self._views_per_burn - 1)
            back_view_idx = random.randint(0, self._views_per_burn - 1)
        else:
            front_view_idx = idx % self._pairs_per_human % self._pairs_per_burn // self._views_per_burn
            back_view_idx = idx % self._pairs_per_human % self._pairs_per_burn % self._views_per_burn
        res = {"label": self.labels[human_idx, burn_idx]}
        if self.return_full_masks:
            res["full_masks"] = (
                read_image(os.path.join(self.root_dir, "full_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "full_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        if self.return_burn_masks:
            res["burn_masks"] = (
                read_image(os.path.join(self.root_dir, "burn_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "burn_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        if self.transform is not None:
            return self.transform(res, self.train_mode)
        else:
            return res

def data_collator(inputs,*kwargs):
    with torch.no_grad():
        front_full_mask = []
        back_full_mask = []
        front_burn_mask = []
        back_burn_mask = []
        labels = []

        for input in inputs:
            front_full_mask.append(input["full_masks"][0])
            back_full_mask.append(input["full_masks"][1])
            front_burn_mask.append(input["burn_masks"][0])
            back_burn_mask.append(input["burn_masks"][1])
            labels.append(input["label"])

        front_full_mask = torch.stack(front_full_mask, dim=0)
        back_full_mask = torch.stack(back_full_mask, dim=0)
        front_burn_mask = torch.stack(front_burn_mask, dim=0)
        back_burn_mask = torch.stack(back_burn_mask, dim=0)
        labels = torch.stack(labels, dim=0)
        return {
            "front_full_mask": front_full_mask,
            "back_full_mask": back_full_mask,
            "front_burn_mask": front_burn_mask,
            "back_burn_mask": back_burn_mask,
            "labels": labels
        }

In [3]:
from torch.utils.data import Subset

train_set = SAM_Precessed_MHB("sam_precessed_MHB", train_mode=True)
test_set = SAM_Precessed_MHB("sam_precessed_MHB")

train_indices = torch.tensor([list(range(i*625*train_set._pairs_per_human,(i*625+500)*train_set._pairs_per_human)) for i in range(8)]).flatten()
test_indices = torch.tensor([list(range((i*625+500)*test_set._pairs_per_human,(i*625+625)*test_set._pairs_per_human)) for i in range(8)]).flatten()

train_set = Subset(train_set, train_indices)
test_set = Subset(test_set, test_indices)

In [4]:
class BurnAreaNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone_a = torch.nn.Sequential(
            torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten()
        )
        self.backbone_b = torch.nn.Sequential(
            torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten()
        )
        self.head = torch.nn.Sequential(
            torch.nn.Linear(128, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 1),
            torch.nn.Sigmoid()
        )

    def forward(self, front_full_mask, back_full_mask, front_burn_mask, back_burn_mask):
        features = torch.cat([self.backbone_a(front_full_mask), self.backbone_a(back_full_mask)], dim=-1) - torch.cat([self.backbone_b(front_burn_mask), self.backbone_b(back_burn_mask)], dim=-1)
        return self.head(features)

In [5]:
from torchvision.transforms import v2 as T

transform = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.uint8, scale=True),
    T.Resize((224, 224)),
    T.ToDtype(torch.float32,scale=True)
])

class HFCompatibleModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, front_full_mask, back_full_mask, front_burn_mask, back_burn_mask, labels=None, **kwargs):
        pre = self.model(transform(front_full_mask), transform(back_full_mask), transform(front_burn_mask), transform(back_burn_mask))[:, 0]
        if labels is not None:
            loss = (pre - labels).abs().mean()
            return {"loss": loss, "logits": pre}
        else:
            return {"logits": pre}

In [6]:
import time
import transformers

trainer_args = transformers.TrainingArguments(
    num_train_epochs=480,
    output_dir="./logs/" + time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime()),
    dataloader_num_workers=8,
    per_device_train_batch_size=128,
    weight_decay=1e-3,
    logging_strategy="epoch",
    remove_unused_columns=False,
    save_strategy="epoch",
    save_total_limit=1,
    metric_for_best_model="loss",
    greater_is_better=False,
    bf16=True,
    bf16_full_eval=True,
)

compatible_model = HFCompatibleModel(BurnAreaNet()).cuda()

In [7]:
# trainer = transformers.Trainer(
#     model = compatible_model,
#     train_dataset=train_set,
#     data_collator=data_collator,
#     args=trainer_args,
# )

# trainer.train()

In [8]:
from safetensors.torch import load_file

compatible_model.load_state_dict(load_file("./sam_p_MHB_model_cbrp.safetensors"))

<All keys matched successfully>

In [9]:
from torch.utils.data import DataLoader
from tqdm import tqdm

test_dataloader = DataLoader(test_set, batch_size=128, num_workers=8, collate_fn=data_collator)
compatible_model.eval()
pre_list = []
label_list = []
with torch.no_grad():
    for batch in tqdm(test_dataloader):
        output = compatible_model(batch["front_full_mask"].cuda(), batch["back_full_mask"].cuda(), batch["front_burn_mask"].cuda(), batch["back_burn_mask"].cuda())
        pre_list.append(output["logits"].cpu())
        label_list.append(batch["labels"])

pre_list = torch.cat(pre_list).reshape(1000,8,16)
label_list = torch.cat(label_list).reshape(1000,8,16)

100%|██████████| 1000/1000 [01:20<00:00, 12.49it/s]


In [10]:
from sklearn.metrics import r2_score
print("R2 Score:", r2_score(label_list.flatten(), pre_list.flatten()))

R2 Score: 0.9953035116195679


In [11]:
from sklearn.metrics import classification_report

triclass_label = torch.zeros_like(label_list)
triclass_label[label_list < 0.05] = 0
triclass_label[(label_list >= 0.05) & (label_list < 0.2)] = 1
triclass_label[label_list >= 0.2] = 2
triclass_pre = torch.zeros_like(pre_list)
triclass_pre[pre_list < 0.05] = 0
triclass_pre[(pre_list >= 0.05) & (pre_list < 0.2)] = 1
triclass_pre[pre_list >= 0.2] = 2
print(classification_report(triclass_label.flatten(), triclass_pre.flatten(), digits=3))

              precision    recall  f1-score   support

         0.0      0.942     0.933     0.937     15520
         1.0      0.936     0.939     0.938     32240
         2.0      0.987     0.987     0.987     80240

    accuracy                          0.969    128000
   macro avg      0.955     0.953     0.954    128000
weighted avg      0.969     0.969     0.969    128000



In [12]:
mae = (pre_list-label_list).abs()
print(f"MAE:{mae.mean().item()*100:.2f}, STD:{torch.std(mae).item()*100:.2f}")

MAE:1.29, STD:1.39


In [13]:
mre = (pre_list-label_list).abs()/(label_list)
print(f"MRE:{mre.mean().item()*100:.2f}, STD:{torch.std(mre).item()*100:.2f}")

MRE:11.51, STD:123.36


In [14]:
import pandas as pd
import pingouin as pg

reshaped_pre = pre_list.view(-1, test_set.dataset._pairs_per_burn)
n_subjects, n_raters = reshaped_pre.shape

long_data_list = []
for i in range(n_raters):
    rater_name = f'Observer_{i+1}'
    temp_df = pd.DataFrame({
        'subject': torch.arange(n_subjects),
        'rater': rater_name,
        'rating': reshaped_pre[:, i] * 100
    })
    long_data_list.append(temp_df)

long_data = pd.concat(long_data_list, ignore_index=True)
pg.intraclass_corr(
    data=long_data,
    targets='subject',
    raters='rater',
    ratings='rating'
)

,Type,Description,ICC,F,df1,df2,pval,CI95
0,ICC1,Single raters absolute,0.998617,11555.034688,7999,120000,0.0,"[1.0, 1.0]"
1,ICC2,Single random raters,0.998617,11555.723584,7999,119985,0.0,"[1.0, 1.0]"
2,ICC3,Single fixed raters,0.998617,11555.723584,7999,119985,0.0,"[1.0, 1.0]"
3,ICC1k,Average raters absolute,0.999913,11555.034688,7999,120000,0.0,"[1.0, 1.0]"
4,ICC2k,Average random raters,0.999913,11555.723584,7999,119985,0.0,"[1.0, 1.0]"
5,ICC3k,Average fixed raters,0.999913,11555.723584,7999,119985,0.0,"[1.0, 1.0]"


In [15]:
baseline_pre = torch.load("sam_p_MHB_baseline_pre.pt")
baseline_mae = (baseline_pre - label_list).abs()
baseline_mre = ((baseline_pre - label_list).abs()/label_list)
print(pg.ttest((baseline_mae.view(-1)*100).numpy().tolist(), (mae.view(-1)*100).numpy().tolist(), paired=True))
print(pg.ttest((baseline_mre.view(-1)*100).numpy().tolist(), (mre.view(-1)*100).numpy().tolist(), paired=True))

                 T     dof alternative  p_val          CI95   cohen_d  power  \
T_test  128.302461  127999   two-sided    0.0  [1.17, 1.21]  0.444753    1.0   

       BF10  
T_test  inf  
               T     dof alternative     p_val           CI95   cohen_d  \
T_test  1.091809  127999   two-sided  0.274919  [-0.29, 1.02]  0.004162   

           power   BF10  
T_test  0.319135  0.006  


In [16]:
for i in range(8):
    print(f"Burn Level {i+1}:", end=" ")
    print(f"MAE: {mae[:, i].mean().item()*100:.2f}±{torch.std(mae[:, i]).item()*100:.2f}, ", end=" / ")
    print(f"{baseline_mae[:, i].mean().item()*100:.2f}±{torch.std(baseline_mae[:, i]).item()*100:.2f}, ", end="")
    print(f"MRE: {mre[:, i].mean().item()*100:.2f}±{torch.std(mre[:, i]).item()*100:.2f}", end=" / ")
    print(f"{baseline_mre[:, i].mean().item()*100:.2f}±{torch.std(baseline_mre[:, i]).item()*100:.2f}")

Burn Level 1: MAE: 0.45±0.35,  / 0.62±0.56, MRE: 56.23±345.34 / 28.94±24.56
Burn Level 2: MAE: 0.74±0.57,  / 1.27±0.98, MRE: 9.94±7.74 / 16.95±12.80
Burn Level 3: MAE: 1.08±0.85,  / 1.95±1.57, MRE: 7.30±5.89 / 13.13±10.45
Burn Level 4: MAE: 1.39±1.08,  / 2.58±2.21, MRE: 5.57±4.34 / 10.31±8.76
Burn Level 5: MAE: 1.57±1.22,  / 2.95±2.62, MRE: 4.50±3.47 / 8.47±7.55
Burn Level 6: MAE: 1.67±1.38,  / 3.26±3.37, MRE: 3.70±3.04 / 7.23±7.45
Burn Level 7: MAE: 1.91±1.87,  / 3.59±4.31, MRE: 3.11±3.04 / 5.84±7.10
Burn Level 8: MAE: 1.48±2.11,  / 3.59±6.74, MRE: 1.73±2.60 / 4.14±7.59


In [17]:
import json

with open("mass_human_burns/gender.json", "r") as f:
    gender_data = json.load(f)

gender_mask = torch.zeros(5000, dtype=bool)
gender_mask[torch.tensor(gender_data["male"])]=1

test_human_indices = (test_indices // test_set.dataset._pairs_per_human).view([1000,8,16])[:,0,0]
test_gender_mask = gender_mask[test_human_indices]

print(f"Number of males: {test_gender_mask.sum().item()}, Number of females: {(~test_gender_mask).sum().item()}")

print(f"Male:", end=" ")
print(f"MAE: {mae[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(mae[test_gender_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mae[test_gender_mask==1]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(mre[test_gender_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mre[test_gender_mask==1]).item()*100:.2f}")
print(f"Female:", end=" ")
print(f"MAE: {mae[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(mae[test_gender_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mae[test_gender_mask==0]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(mre[test_gender_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mre[test_gender_mask==0]).item()*100:.2f}")

print(pg.ttest((mae[test_gender_mask==0].view(-1)*100).numpy().tolist(), (mae[test_gender_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[test_gender_mask==0].view(-1)*100).numpy().tolist(), (mre[test_gender_mask==1].view(-1)*100).numpy().tolist()))

Number of males: 493, Number of females: 507
Male: MAE: 1.29±1.37 / 2.49±3.93, MRE: 9.92±58.93 / 11.87±14.17
Female: MAE: 1.28±1.40 / 2.46±3.08, MRE: 13.06±163.19 / 11.88±14.32
              T            dof alternative     p_val          CI95   cohen_d  \
T_test -1.57542  127997.385024   two-sided  0.115162  [-0.03, 0.0]  0.008805   

           power   BF10  
T_test  0.350277  0.022  
               T           dof alternative     p_val          CI95   cohen_d  \
T_test  4.606203  81952.817081   two-sided  0.000004  [1.81, 4.48]  0.025476   

           power     BF10  
T_test  0.995296  254.845  


In [18]:
age_mask = torch.zeros(1000, dtype=bool)
age_mask[:500] = 1

print(f"Adult:", end=" ")
print(f"MAE: {mae[age_mask==1].mean().item()*100:.2f}±{torch.std(mae[age_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[age_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mae[age_mask==1]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[age_mask==1].mean().item()*100:.2f}±{torch.std(mre[age_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[age_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mre[age_mask==1]).item()*100:.2f}")
print(f"Child:", end=" ")
print(f"MAE: {mae[age_mask==0].mean().item()*100:.2f}±{torch.std(mae[age_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[age_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mae[age_mask==0]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[age_mask==0].mean().item()*100:.2f}±{torch.std(mre[age_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[age_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mre[age_mask==0]).item()*100:.2f}")

print(pg.ttest((mae[age_mask==0].view(-1)*100).numpy().tolist(), (mae[age_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[age_mask==0].view(-1)*100).numpy().tolist(), (mre[age_mask==1].view(-1)*100).numpy().tolist()))

Adult: MAE: 1.26±1.42 / 2.42±3.10, MRE: 10.36±66.23 / 11.51±13.53
Child: MAE: 1.31±1.34 / 2.53±3.90, MRE: 12.67±161.38 / 12.25±14.93
               T     dof alternative         p_val          CI95   cohen_d  \
T_test  5.965686  127998   two-sided  2.442455e-09  [0.03, 0.06]  0.033349   

           power       BF10  
T_test  0.999969  3.358e+05  
               T     dof alternative     p_val          CI95   cohen_d  \
T_test  3.347912  127998   two-sided  0.000814  [0.96, 3.66]  0.018715   

          power   BF10  
T_test  0.91742  1.712  


In [19]:
for i in range(4):
    pose_mask = torch.zeros(1000, dtype=bool)
    pose_mask[i*125:(i+1)*125] = 1 # Adult
    pose_mask[i*125+500:(i+1)*125+500] = 1 # Child
    print(f"Pose {i+1}:", end=" ")
    print(f"MAE: {mae[pose_mask==1].mean().item()*100:.2f}±{torch.std(mae[pose_mask==1]).item()*100:.2f}", end=" / ")
    print(f"{baseline_mae[pose_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mae[pose_mask==1]).item()*100:.2f}", end=" / ")
    print(f"MRE: {mre[pose_mask==1].mean().item()*100:.2f}±{torch.std(mre[pose_mask==1]).item()*100:.2f}", end=" / ")
    print(f"{baseline_mre[pose_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mre[pose_mask==1]).item()*100:.2f}") 

Pose 1: MAE: 1.40±1.52 / 2.17±1.92 / MRE: 9.80±42.90 / 10.74±12.55
Pose 2: MAE: 1.39±1.65 / 2.44±2.16 / MRE: 10.92±71.82 / 11.55±12.31
Pose 3: MAE: 1.18±1.15 / 2.55±4.18 / MRE: 10.83±64.49 / 12.41±15.34
Pose 4: MAE: 1.17±1.13 / 2.74±4.87 / MRE: 14.50±222.93 / 12.80±16.29


In [20]:
pose_1_mask = torch.zeros(1000, dtype=bool)
pose_1_mask[:125] = 1
pose_1_mask[500:625] = 1
pose_4_mask = torch.zeros(1000, dtype=bool)
pose_4_mask[375:500] = 1
pose_4_mask[875:1000] = 1

print(pg.ttest((mae[pose_1_mask==1].view(-1)*100).numpy().tolist(), (mae[pose_4_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[pose_1_mask==1].view(-1)*100).numpy().tolist(), (mre[pose_4_mask==1].view(-1)*100).numpy().tolist()))

                T    dof alternative          p_val          CI95  cohen_d  \
T_test  21.666601  63998   two-sided  1.002117e-103  [0.21, 0.25]  0.17129   

        power       BF10  
T_test    1.0  3.113e+99  
               T    dof alternative     p_val            CI95   cohen_d  \
T_test -3.701253  63998   two-sided  0.000215  [-7.18, -2.21]  0.029261   

           power   BF10  
T_test  0.959179  8.398  
